# Teton Classifier Notebook

Random Forest Classifier for the Teton Range, based off of Yang et al. (2023)

## Imports and initializations

In [1]:
import os
import ee
import geemap
import pandas as pd


In [2]:
ee.Authenticate()

True

In [3]:
EE_PROJECT = "teton-classifier"
# Alpine-ish mask threshold for this first pass. Adjust after inspecting the map.
ALPINE_MIN_ELEV_M = 2200

START_DATE = f"2018-01-01"
END_DATE = f"2025-12-31"
SCALE = 10
SEED = 40

if EE_PROJECT:
    ee.Initialize(project=EE_PROJECT)
else:
    ee.Initialize()
print("Earth Engine initialized.")

Earth Engine initialized.


## Map, draw study boundary or use preset box


In [5]:
Map = geemap.Map(center=[43.75, -110.75], zoom=10)
aoi = ee.Geometry.Rectangle([-111.05, 43.35, -110.45, 44.05])
Map.addLayer(aoi, {"color": "yellow"}, "AOI")
Map

Map(center=[43.75, -110.75], controls=(WidgetControl(options=['position', 'transparent_bg'], position='toprigh…

## Gather data

In [6]:
features = [
    # Sentinel-2 reflectance
    "B2", "B3", "B4",      # blue, green, red
    "B8",                 # near infrared
    "B11", "B12",         # shortwave infrared

    # Spectral indices
    "NDVI",               # vegetation greenness
    "NDWI",               # water / wetness
    "NDSI",               # snow / ice
    "BSI",                # bare soil / rock tendency

    # Terrain
    "elevation",
    "slope",
    "aspect"
]

## Alternative approach: count snow pixel years

### Get terrain

In [26]:
terrain = ee.Algorithms.Terrain(
    ee.Image("NASA/NASADEM_HGT/001").select("elevation")
).clip(aoi)

elevation = terrain.select("elevation")
slope = terrain.select("slope")
aspect = terrain.select("aspect")

### Generate yearly ndsi, ndwi gathering functions

In [27]:
def mask_s2_clouds(image):
    qa = image.select("QA60")


    # Identifying relevant bits in the QA60 band for cloud and cirrus detection
    cloud_bit = 1 << 10
    cirrus_bit = 1 << 11

    mask = (
        qa.bitwiseAnd(cloud_bit).eq(0)
        .And(qa.bitwiseAnd(cirrus_bit).eq(0))
    )

    return image.updateMask(mask).divide(10000)



def late_summer_snow_mask(year):
    s2_year = (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(aoi)
        .filterDate(f"{year}-08-15", f"{year}-09-30")
        .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 70))
        .map(mask_s2_clouds)
        .median()
        .clip(aoi)
    )

    ndsi = s2_year.normalizedDifference(["B3", "B11"]).rename("NDSI")
    ndwi = s2_year.normalizedDifference(["B3", "B8"]).rename("NDWI")

    snow = (
        ndsi.gt(0.4)
        .And(s2_year.select("B3").gt(0.15))
        .And(ndwi.lt(0.4))
        .And(elevation.gt(2200))
        .rename("snow")
    )

    return snow.toByte()

### Collect images

In [35]:
snow_collection = ee.ImageCollection([
    late_summer_snow_mask(year) for year in range(2018, 2025)
])

snow_year_count = snow_collection.sum().rename("snow_year_count")
persistent_snow = snow_year_count.gte(2).rename("persistent_snow")

### Map

In [36]:
Map.addLayer(
    snow_year_count,
    {
        "min": 0,
        "max": len(list(range(2018, 2025))),
        "palette": ["black", "blue", "cyan", "white"]
    },
    "Late-summer snow year count",
    opacity=0.7
)
\
Map.add_tile_layer(
    url="https://basemap.nationalmap.gov/arcgis/rest/services/USGSTopo/MapServer/tile/{z}/{y}/{x}",
    name="USGS Topo",
    attribution="USGS The National Map",
    opacity=0.45,
)
Map.addLayer(
    persistent_snow.selfMask(),
    {"palette": ["white"]},
    "Persistent snow"
)
Map

Map(bottom=3061503.0, center=[43.661321178702764, -110.83132266998292], controls=(WidgetControl(options=['posi…

# Export

In [37]:
task = ee.batch.Export.image.toDrive(
    image=persistent_snow.toByte().clip(aoi),
    description="persistent_snow_mask",
    folder="gee_exports",
    fileNamePrefix="persistent_snow_mask",
    region=aoi,
    scale=10,
    maxPixels=1e13
)

task.start()

print("Started export task:", task.id)

Started export task: EKLKDSUCU7VSKNTFKCI7RPJ7


In [43]:
task.status()

{'state': 'RUNNING',
 'description': 'persistent_snow_mask',
 'priority': 100,
 'creation_timestamp_ms': 1782592489609,
 'update_timestamp_ms': 1782592497078,
 'start_timestamp_ms': 1782592497006,
 'task_type': 'EXPORT_IMAGE',
 'attempt': 1,
 'id': 'EKLKDSUCU7VSKNTFKCI7RPJ7',
 'name': 'projects/teton-classifier/operations/EKLKDSUCU7VSKNTFKCI7RPJ7'}